# Chapter 3 · Vimentin–Microtubule Bonds — Dealing with Selection Bias

Section 3.3 (pp. 38–45), Figures 10–13.

| Figure | Page | Content |
|---|---|---|
| Figure 10 | p. 38 | Influence of the selection bias (three rows) |
| Figure 11 | p. 40 | Mitigation 1: dropping data |
| Figure 12 | p. 43 | Neural-network surrogate of the stress–strain curve |
| Figure 13 | p. 44 | Surrogate performance vs. training-set size |

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic_2d
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # the MLPs are trained for a fixed 500 iterations

import lambert_thesis as lt
from lambert_thesis import bonds, surrogate, plotting
from lambert_thesis.plotting import savefig

plotting.setup()
rng = lt.set_seed(lt.SEED)

RU0_GT, XU_GT = 0.05, 0.2     # ground truth (section 3.2)
DT = 1e-2
V = 0.2                       # constant driving velocity
force = bonds.force_curve(50)

ru0_grid = np.linspace(0, 2.2 * RU0_GT, 100)
xu_grid = np.linspace(0, 2.2 * XU_GT, 100)

def posterior(trajs, censor_below=None):
    return bonds.normalize_log(bonds.log_likelihood_grid(trajs, ru0_grid, xu_grid, DT, censor_below=censor_below))

Obtaining experimental data is difficult: the data is noisy, and the formation of an IF–MT bond is hard to notice early
on. A bond is only identified by a sufficiently high force — the lab protocol accepts a trajectory as bonded only if it
exceeds 2 pN at its peak. This cut-off **biases** the inference.

The data are the 50 trajectories at constant velocity of Figure 7.
A trajectory is *censored* (ignored) if its maximum force stays below a threshold $F_\text{thr}$.
The two thresholds below correspond to the force reached after 400 and 490 time steps.

*Code:* censoring is a one-line filter on the list of trajectories; the posterior is the grid posterior from 02a.
`plotting.grid_posterior` draws it with marginals (teal) and posterior means (green lines).

In [ ]:
trajs, _ = bonds.simulate_trajectories(force, RU0_GT, XU_GT, v=V, dt=DT, rng=rng, n=50)

thresholds = [None, float(force(V * 400 * DT)), float(force(V * 490 * DT))]
print("thresholds F_thr:", [None if t is None else round(t, 2) for t in thresholds])

def bias_row(F_thr):
    kept = [F for F in trajs if F_thr is None or F.max() > F_thr]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1, 1.1]})
    for i, F in enumerate(trajs):
        used = F_thr is None or F.max() > F_thr
        a1.plot(F + i, color="k" if used else "r")
    a1.plot([], [], "k", label="Used Data")
    if F_thr is not None:
        a1.plot([], [], "r", label=f"Ignored Data (peak force < {F_thr:.2f})")
    a1.legend(loc="upper left", frameon=False)
    a1.set(title="Simulated Trajectories (offset for visibility)", xlabel="Timesteps", ylabel="Force")
    plotting.grid_posterior(a2, posterior(kept), ru0_grid, xu_grid, truth=(RU0_GT, XU_GT), marginals=True,
                            title=f"Posteriors, {len(kept)} trajectories, Dashed: 95th percentile")
    fig.tight_layout()
    return fig

### Figure 10 (p. 38), top row: all trajectories are used — the posterior captures the ground truth.

In [ ]:
bias_row(thresholds[0]); savefig("fig10a_bias_all_data");

### Figure 10, middle row: trajectories with low peak force are censored (red).

In [ ]:
bias_row(thresholds[1]); savefig("fig10b_bias_threshold_low");

### Figure 10, bottom row: higher threshold.

In [ ]:
bias_row(thresholds[2]); savefig("fig10c_bias_threshold_high");

**Figure 10 (p. 38).** Requiring a minimum peak force censors trajectories (red). This biases the posterior,
which no longer recovers the ground truth (red cross). Intuitively, censoring removes unbinding events at low forces,
which suggests a lower base unbinding rate $r_{u,0}$ — the posterior mass drifts to the left.

## 3.3.1 Mitigation 1: Dropping data

Censoring only compromises the data *below* the threshold. The ratio of bound and unbound time segments above the
threshold is unaffected, so dropping **all data points below the threshold** removes the bias. We choose a threshold
that censors about 25 % of the trajectories (here: the force after 500 time steps) and generate 500 trajectories.

*Code:* `bonds.log_likelihood_grid(..., censor_below=F_thr)` drops every data point before a trajectory first
exceeds `F_thr`; trajectories that never exceed it are discarded completely.

In [ ]:
trajs500, _ = bonds.simulate_trajectories(force, RU0_GT, XU_GT, v=V, dt=DT, rng=rng, n=500)
F_thr = float(force(V * 500 * DT))
observed = [F for F in trajs500 if F.max() > F_thr]
print(f"threshold {F_thr:.2f}: {len(observed)} of {len(trajs500)} trajectories observed "
      f"({1 - len(observed) / len(trajs500):.0%} censored)")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5), gridspec_kw={"width_ratios": [1, 1.1]})
for i, F in enumerate(trajs500):
    if F.max() <= F_thr:
        a1.plot(F + i, color="r", alpha=0.5, lw=0.8)
    else:
        k = np.argmax(F > F_thr)
        a1.plot(np.arange(k + 1), F[:k + 1] + i, color="0.3", alpha=0.5, lw=0.8)
        a1.plot(np.arange(k, len(F)), F[k:] + i, color="g", lw=0.8)
a1.axvline(500, color="C0", lw=3)
a1.plot([], [], "r", label="unobserved (threshold never reached)")
a1.plot([], [], color="0.3", label="observed, but dropped (before threshold)")
a1.plot([], [], "g", label="used for the inference")
a1.plot([], [], color="C0", lw=3, label="time at which the threshold force is reached")
a1.legend(loc="upper left", fontsize=7, framealpha=0.9)
a1.set(xlabel="Time", ylabel="Force")
plotting.grid_posterior(a2, posterior(trajs500, censor_below=F_thr), ru0_grid, xu_grid, truth=(RU0_GT, XU_GT),
                        marginals=True, title=f"Posteriors, {len(observed)} trajectories, Dashed: 95th percentile")
fig.tight_layout()
savefig("fig11_bias_drop_data");

**Figure 11 (p. 40).** Red trajectories are unobserved, as they never reach the threshold before the bond breaks.
Dropping all data points before the threshold leaves only data unaffected by the censoring (green). The posterior is
unbiased again, but since no data at low forces remains, the base unbinding rate is not inferred well — even with
500 trajectories.

## 3.3.2 Mitigation 2: Rejection modelling (overview)

A smarter way is to include the rejection explicitly in the generative model: proposals $T^*$ are generated and rejected
with probability $p_r(T^*)$ (here: rejected if $\max_i F_i < F_\text{thr}$) until one is accepted. Summing over all
pathways that lead to an accepted trajectory $T$ gives

$$\mathcal L_r(T)=\mathcal L(T)\sum_{i=0}^\infty P_r^i=\frac{\mathcal L(T)}{1-P_r},$$

with the marginal rejection probability $P_r$, obtained by a Monte-Carlo integral over the rejection-free model for every
set of bond parameters. The rejection model is demonstrated on the Cytosim data in notebook 02d.

The Monte-Carlo integral needs a representative force curve. Instead of the theoretical model, the force–strain curve
can be learned from observations as a fast **surrogate**: a network with 100 hidden ReLU nodes predicts the force
from elapsed time and driving velocity. The unbinding model stays the Bell–Evans rate.

*Code:* `surrogate.fit_force_surrogate(trajs, velocities, dt, activation=...)` — a scikit-learn `MLPRegressor`
(100 hidden units, 500 training iterations) behind a `StandardScaler`. Inputs are the time, the driving velocity and
the strain $v\cdot t$ derived from both. The networks of Figure 12 use tanh activations. Training trajectories use equally spaced velocities in $[0.01, 0.4]$;
a test set of 1000 trajectories with $v\sim\mathcal U(0.01, 0.4)$ is used to score the prediction ($R^2$).

In [ ]:
test_trajs, test_v = bonds.simulate_trajectories(force, RU0_GT, XU_GT, v=lambda r: r.uniform(0.01, 0.4),
                                                 dt=DT, rng=rng, n=1000)
X_test, y_test = surrogate.features(test_trajs, test_v, DT)

def train(n, seed=lt.SEED):
    v = np.linspace(0.01, 0.4, n)
    tr, _ = bonds.simulate_trajectories(force, RU0_GT, XU_GT, v=v, dt=DT, rng=rng, n=n)
    return surrogate.fit_force_surrogate(tr, v, DT, seed=seed, activation="tanh"), tr, v

def surrogate_figure(n, n_show=20):
    model, tr, v = train(n)
    pred = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    t_idx, v_feat = X_test[:, 0] / DT, X_test[:, 1]
    fig, ax = plt.subplots(3, 1, figsize=(7, 13))
    # top: prediction on unseen trajectories (concatenated)
    m = sum(len(F) for F in test_trajs[:n_show])
    ax[0].plot(pred[:m], "r", label="prediction of the surrogate")
    ax[0].plot(y_test[:m], "k--", label=f"data ({n_show} unseen test trajectories, concatenated)")
    ax[0].set(title=f"Trained on {n} trajectories (R2={r2:.3g})", ylabel="Force", xlabel="Sample index (concatenated)")
    ax[0].legend(fontsize=8)
    # middle: observed F(t, v) on the test set + training data; bottom: prediction (+ inset: relative error)
    ext = [t_idx.min(), t_idx.max(), v_feat.min(), v_feat.max()]
    for a, values, title in [(ax[1], y_test, "Observed F(v,t)"), (ax[2], pred, "Predicted F(v,t)")]:
        stat = binned_statistic_2d(t_idx, v_feat, values, bins=100).statistic
        with np.errstate(invalid="ignore", divide="ignore"):
            im = a.imshow(np.log(stat).T, origin="lower", extent=ext, aspect="auto")
        a.set(title=title, xlabel="Sample index", ylabel="v")
        a.grid(False)
    fig.colorbar(im, ax=ax[1:], label="log force", shrink=0.6)
    for F, vi in zip(tr, v):
        ax[1].plot([0, len(F)], [vi, vi], "r", lw=1.2)
    ax[1].plot([], [], "r", label="training trajectories (length = duration)")
    ax[1].legend(loc="upper right", fontsize=8)
    ins = ax[2].inset_axes([0.5, 0.35, 0.45, 0.6])
    err = binned_statistic_2d(t_idx, v_feat, np.abs(y_test - pred) / np.abs(pred), bins=100).statistic
    with np.errstate(invalid="ignore", divide="ignore"):
        ins.imshow(np.log(err).T, origin="lower", extent=ext, aspect="auto")
    ins.set_title("Error (log relative)", fontsize=8); ins.set_xticks([]); ins.set_yticks([])
    return fig, r2

### Figure 12 (p. 43), left column: surrogate trained on 5 trajectories

In [ ]:
fig, r2_5 = surrogate_figure(5); savefig("fig12a_surrogate_5_trajectories");

### Figure 12 (p. 43), right column: surrogate trained on 30 trajectories

In [ ]:
fig, r2_30 = surrogate_figure(30); savefig("fig12b_surrogate_30_trajectories");

**Figure 12 (p. 43).** Learning of the stress–strain curve. Top: prediction overlaid on unseen force data.
Middle: observed force as a function of time and velocity (test set; white = no data), red bars = training data.
Bottom: model prediction. The model only predicts the magnitude of the force, not when the bond breaks.

For Figure 13 the surrogate (ReLU activations) is trained for every training-set size $n=2,\dots,29$ ten times, each
time on newly simulated trajectories; the best of the ten $R^2$ scores is reported. Every $n$ is scored on its own
freshly simulated test set of 1000 trajectories (cached in `results/`).

In [ ]:
def performance_scores(n_tries=10):
    ns = np.arange(2, 30)
    best = []
    for n in ns:
        te, tv = bonds.simulate_trajectories(force, RU0_GT, XU_GT, v=lambda r: r.uniform(0.01, 0.4), dt=DT, rng=rng, n=1000)
        Xt, yt = surrogate.features(te, tv, DT)
        v = np.linspace(0.01, 0.4, n)
        scores_n = []
        for _ in range(n_tries):
            tr, _ = bonds.simulate_trajectories(force, RU0_GT, XU_GT, v=v, dt=DT, rng=rng, n=n)
            model = surrogate.fit_force_surrogate(tr, v, DT, seed=int(rng.integers(2**31)))
            scores_n.append(model.score(Xt, yt))
        best.append(max(scores_n))
    return {"n": ns, "best_r2": np.array(best)}

res = lt.cached("fig13_surrogate_scores", performance_scores)
ns, scores = res["n"], res["best_r2"]

plt.figure()
plt.plot(ns, scores, "o-", label="best R² of 10 trainings (test set: 1000 trajectories)")
plt.axhline(1, color="k", ls="--", label="perfect prediction")
plt.ylim(0, 1.1)
plt.xlabel("Number of training trajectories")
plt.ylabel("R2 Score")
plt.legend(loc="lower right")
savefig("fig13_surrogate_performance");

**Figure 13 (p. 44).** Surrogate performance vs. training-set size. $R^2=1$ is exact reproduction; $R^2=0$ would mean
the model only predicts the mean. Because the force curves are simple, a few examples suffice.

Rejection modelling avoids dropping data, but needs a generative model and the integral $P_r$ for every set of bond
parameters — more computation instead of more experiments.